**Purpose:** `v5.2.X_news.ipynb` — part of `03_portfolio/LSTM_returns` (see the stage README).

**Outputs:** `LSTMvia-returns_news_weights.npy`, `test_bl_returns.csv`, `test_bl_weights.csv`, `{prefix}_cumulative_returns.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_NEWS, SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_NEWS_BL


In [ ]:
from src.config import PROJECT_ROOT


# LSTM + Black–Litterman — Sentiment Feature Extension (v5.3)

Extends v5.2.2 by adding GDELT-based sentiment features per asset.

**Strategy**
- Freeze the **top-N feature sets** from the v5.2.2 Optuna study (each trial keeps its own feature picks)
- Re-tune **all hyperparameters** (architecture + training) alongside the new sentiment search space
- Per-trial: one base LSTM config from top-N is randomly sampled, then sentiment params are tuned on top

**New Optuna search space**
| Parameter | Choices |
|---|---|
| `sentiment_live_group` | `balance` \| `balance_ratio` \| `count_positive+count_negative` \| `ratio_positive+ratio_negative` |
| `sentiment_hist_group` | `none` \| `balance_ratio_roll22` \| `balance_ratio_roll3` \| `balance_ratio_roll5` \| `balance_ratio_t-1+balance_ratio_t-5` \| `balance_ratio_variation1+balance_ratio_variation5` |
| `sentiment_threshold` | 0.00, 0.50, 0.75, 0.90 |
| `news_nan_strategy` | zero_fill, carry_forward, mask_asset |

**Live group is mandatory** (always exactly one). **Historical group is optional** (`none` = skip).
Paired indicators (same line in the spec) always enter as a group.

**`TOP_N_BASE`** is derived automatically from `best_params['top_n']` of the v5.2.2 BL study.

Plus all LSTM/training hyperparameters are re-tuned.

`_NEWS_NAN` is a **single global flag** (not per-asset) — 1 on days the news feed was down entirely, 0 otherwise.
It is appended as an `identity`-transformed feature, broadcast to all N assets in `X_raw`.

**Sentiment column naming in df**
```
{ASSET}_news_threshold{THRESHOLD}_{INDICATOR}   ← per-asset sentiment values
_NEWS_NAN                                        ← single global availability flag
```
e.g. `XLB_news_threshold0.50_balance_ratio`, `_NEWS_NAN`


## 1) Imports

In [ ]:
# Copy all imports from v5.2.2 — no new deps needed
from __future__ import annotations

import os
import json
import copy
import random
import logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

## 2) Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
DATASET_PATH  = str(PROJECT_ROOT / "03_portfolio/dataset.parquet")
SPREADS_PATH  = str(PROJECT_ROOT / "01_data/aux/bid-ask_spread.json")

# Path to the v5.2.2 Optuna DBs
V522_LSTM_DB_PATH = "v522/joint_asset_lstm.db"
V522_BL_DB_PATH   = "v522/black_litterman_optuna.db"

# ── Universe ─────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Base feature buckets (carried from v5.2.2, used to reconstruct picks) ─
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Sentiment feature catalogue ───────────────────────────────────────────
# NEWS_LIVE: exactly ONE group must be chosen (mandatory)
# Groups that must come together are represented as tuples.
# Single-indicator groups are plain strings for Optuna compatibility.
SENTIMENT_LIVE_GROUPS = [
    "balance",                                  # solo
    "balance_ratio",                            # solo
    ("count_positive", "count_negative"),       # always together
    ("ratio_positive", "ratio_negative"),       # always together
]

# NEWS_HISTORICAL: at most ONE group (optional — Optuna may choose 'none')
SENTIMENT_HIST_GROUPS = [
    "none",                                                         # skip historical
    "balance_ratio_roll22",                                         # solo
    "balance_ratio_roll3",                                          # solo
    "balance_ratio_roll5",                                          # solo
    ("balance_ratio_t-1", "balance_ratio_t-5"),                     # always together
    ("balance_ratio_variation1", "balance_ratio_variation5"),       # always together
]

SENTIMENT_THRESHOLDS = ["0.00", "0.50", "0.75", "0.90"]

# ── Study settings ─────────────────────────────────────────────────────────
OUT_DIR         = 'v522_news'
N_TRIALS        = 0      # trials per LSTM rank (× TOP_N_BASE studies total)
N_TRIALS_BL     = 0      # BL HPO trials
MODEL_SEEDS     = [11, 22, 33]
BL_SEED         = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_NEWS_BL


## 3) Utilities  *(unchanged from v5.2.2)*

In [ ]:
def setup_logger(log_dir: str) -> logging.Logger:
    logger = logging.getLogger(f"sentiment_lstm_{os.path.basename(log_dir)}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s")
    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    logger.propagate = False
    return logger

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_makedirs(path: str):
    os.makedirs(path, exist_ok=True)

## 4) Walk-forward splits  *(unchanged from v5.2.2)*

In [ ]:
def make_walk_forward_splits(dates_index: pd.DatetimeIndex):
    splits = []
    periods = [
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2020-06-30"),
         pd.Timestamp("2020-07-01"), pd.Timestamp("2021-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2021-06-30"),
         pd.Timestamp("2021-07-01"), pd.Timestamp("2022-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2022-06-30"),
         pd.Timestamp("2022-07-01"), pd.Timestamp("2023-06-30")),
    ]
    for tr_start, tr_end, v_start, v_end in periods:
        train_idx = np.where((dates_index >= tr_start) & (dates_index <= tr_end))[0]
        val_idx   = np.where((dates_index >= v_start)  & (dates_index <= v_end))[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            print(f"[SKIP SPLIT] train {tr_start.date()}-{tr_end.date()} "
                  f"val {v_start.date()}-{v_end.date()} | "
                  f"lens train={len(train_idx)} val={len(val_idx)}")
            continue
        splits.append((train_idx, val_idx))
    if len(splits) == 0:
        raise ValueError("No valid walk-forward splits.")
    return splits


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    train_dates  = dates_index[train_idx]
    train_end    = train_dates[-1]
    es_start     = train_end - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)
    es_mask      = train_dates >= es_start
    core_idx     = train_idx[~es_mask]
    es_idx       = train_idx[es_mask]
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut      = max(min_train_days, len(train_idx) - 252 * es_years)
        cut      = min(cut, len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]
    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError("Failed to create non-empty early-stopping split.")
    return core_idx, es_idx

## 5) Load top-N base configs from v5.2.2 study

In [ ]:
def _get_top_n_from_bl_study(bl_db_path: str) -> int:
    """
    Read the best BL trial from the v5.2.2 BL Optuna DB and return its
    top_n value.  This is the same top_n that the BL optimisation found
    to maximise validation Sharpe, so we honour it here rather than
    picking an arbitrary number.
    """
    storage  = f"sqlite:///{bl_db_path}"
    bl_study = optuna.load_study(
        study_name="black_litterman_hpo",
        storage=storage,
    )
    completed = [
        t for t in bl_study.trials
        if t.state == TrialState.COMPLETE
        and t.value is not None
        and np.isfinite(t.value)
    ]
    if not completed:
        raise ValueError("No completed BL trials found in v5.2.2 BL DB.")
    best_bl = max(completed, key=lambda t: t.value)
    top_n   = int(best_bl.params["top_n"])
    print(f"  Best BL trial #{best_bl.number}  val_sharpe={best_bl.value:.4f}  "
          f"→ top_n={top_n}")
    return top_n


def load_top_n_base_configs(
    lstm_db_path: str,
    bl_db_path: str,
) -> Tuple[List[Dict[str, Any]], int]:
    """
    Derive top_n from the best BL trial, then load that many top LSTM
    trials from the v5.2.2 LSTM DB.

    Returns:
      configs : list of dicts with keys
                  'picks', 'per_asset_feats', 'params',
                  'trial_number', 'value'
      top_n   : the integer derived from the BL study
    """
    print("Reading top_n from BL study ...")
    top_n = _get_top_n_from_bl_study(bl_db_path)

    print(f"Loading top-{top_n} LSTM configs from {lstm_db_path} ...")
    storage  = f"sqlite:///{lstm_db_path}"
    study    = optuna.load_study(
        study_name="joint_asset_lstm",
        storage=storage,
    )
    completed = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE
        and t.value is not None
        and np.isfinite(t.value)
    ]
    if not completed:
        raise ValueError("No completed LSTM trials found in v5.2.2 DB.")

    top = sorted(completed, key=lambda t: t.value, reverse=True)[:top_n]

    configs = []
    for t in top:
        picks           = t.user_attrs.get("feature_set", {})
        per_asset_feats = t.user_attrs.get("per_asset_feats", [])
        configs.append({
            "picks":           picks,
            "per_asset_feats": per_asset_feats,
            "params":          dict(t.params),
            "trial_number":    t.number,
            "value":           t.value,
        })
        print(f"  Trial #{t.number:5d}  IC={t.value:.5f}  feats={per_asset_feats}")

    return configs, top_n


# ── Run at notebook startup ───────────────────────────────────────────────
BASE_CONFIGS, TOP_N_BASE = load_top_n_base_configs(
    lstm_db_path=V522_LSTM_DB_PATH,
    bl_db_path=V522_BL_DB_PATH,
)
print(f"Loaded {len(BASE_CONFIGS)} base configs (top_n={TOP_N_BASE} from BL study).")

## 6) Feature engineering — base features  *(unchanged from v5.2.2)*

In [ ]:
def pick_features_from_buckets(trial: optuna.Trial, features: Dict[str, List[str]]) -> Dict[str, Any]:
    picks = {}
    picks["Price"] = "log_ret_1d"
    use_ohlc = trial.suggest_categorical("use_ohlc", [0, 1])
    picks["OHLC"] = ["Open", "High", "Low"] if use_ohlc == 1 else []
    for bucket in ["Trend", "Momentum", "Volume"]:
        picks[bucket] = trial.suggest_categorical(f"pick_{bucket}", features[bucket])
    vol_options  = [f for f in features["Volatility"] if f not in ("BBLower", "BBUpper")]
    vol_options.append("BBands")
    vol_choice   = trial.suggest_categorical("pick_Volatility", vol_options)
    picks["Volatility"] = ["BBLower", "BBUpper"] if vol_choice == "BBands" else vol_choice
    use_market   = trial.suggest_categorical("use_market", [0, 1])
    picks["Market"] = "VIXIndexClose" if use_market == 1 else None
    return picks


def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    feats = ["log_ret_1d"]
    for f in picks.get("OHLC", []):
        feats.append(f)
    for bucket in ["Trend", "Momentum", "Volume", "Volatility"]:
        val = picks[bucket]
        if isinstance(val, list):
            feats.extend(val)
        else:
            feats.append(val)
    out  = []
    seen = set()
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns(
    assets: List[str],
    per_asset_feats: List[str],
    picks: Dict[str, Any],
    sentiment_feats: Optional[List[str]] = None,  # NEW: e.g. ['news_threshold0.50_balance_ratio']
) -> List[str]:
    cols = []
    for a in assets:
        cols.append(f"{a}_Close")
        for f in per_asset_feats:
            if f == "log_ret_1d":
                continue
            cols.append(f"{a}_{f}")
        if sentiment_feats:
            for sf in sentiment_feats:
                cols.append(f"{a}_{sf}")
    # _NEWS_NAN is a single global column (not per-asset)
    if sentiment_feats:
        cols.append("_NEWS_NAN")
    if picks.get("Market", None) is not None:
        cols.append(picks["Market"])
    return cols

## 7) Sentiment feature helpers

In [ ]:
# ── Optuna key helpers ────────────────────────────────────────────────────
# Groups with paired indicators are serialised as 'A+B' for Optuna categoricals.

def _group_to_key(g) -> str:
    """Convert a group (str or tuple) to a stable Optuna categorical key."""
    return "+".join(g) if isinstance(g, tuple) else str(g)


def _key_to_indicators(key: str) -> List[str]:
    """Expand an Optuna key back to a flat list of indicator names."""
    return [] if key == "none" else key.split("+")


def pick_sentiment_features(
    trial: optuna.Trial,
    live_groups:  list = SENTIMENT_LIVE_GROUPS,
    hist_groups:  list = SENTIMENT_HIST_GROUPS,
    thresholds:   List[str] = SENTIMENT_THRESHOLDS,
) -> Dict[str, Any]:
    """
    Sample sentiment configuration from Optuna.

    NEWS_LIVE  — mandatory, exactly one group chosen:
      'balance'  |  'balance_ratio'  |
      'count_positive+count_negative'  |  'ratio_positive+ratio_negative'

    NEWS_HISTORICAL — optional, at most one group:
      'none'  |  'balance_ratio_roll22'  |  'balance_ratio_roll3'  |
      'balance_ratio_roll5'  |
      'balance_ratio_t-1+balance_ratio_t-5'  |
      'balance_ratio_variation1+balance_ratio_variation5'

    One threshold is shared across all active sentiment features.
    Paired indicators (same line in the spec) always enter together.

    Returns dict with:
      'threshold'        : str  e.g. '0.50'
      'live_key'         : str  Optuna categorical
      'hist_key'         : str  Optuna categorical ('none' when skipped)
      'live_indicators'  : List[str]  1–2 indicator names
      'hist_indicators'  : List[str]  0–2 indicator names
      'sentiment_cols'   : List[str]  full column suffixes to look up in df,
                           e.g. ['news_threshold0.50_balance_ratio',
                                 'news_threshold0.50_balance_ratio_t-1',
                                 'news_threshold0.50_balance_ratio_t-5']
    """
    threshold = trial.suggest_categorical("sentiment_threshold", thresholds)

    # NEWS_LIVE: mandatory — pick exactly one group key
    live_keys = [_group_to_key(g) for g in live_groups]
    live_key  = trial.suggest_categorical("sentiment_live_group", live_keys)
    live_indicators = _key_to_indicators(live_key)

    # NEWS_HISTORICAL: optional — 'none' means skip
    hist_keys = [_group_to_key(g) for g in hist_groups]
    hist_key  = trial.suggest_categorical("sentiment_hist_group", hist_keys)
    hist_indicators = _key_to_indicators(hist_key)

    prefix         = f"news_threshold{threshold}"
    sentiment_cols = (
        [f"{prefix}_{ind}" for ind in live_indicators]
        + [f"{prefix}_{ind}" for ind in hist_indicators]
    )

    return {
        "threshold":        threshold,
        "live_key":         live_key,
        "hist_key":         hist_key,
        "live_indicators":  live_indicators,
        "hist_indicators":  hist_indicators,
        "sentiment_cols":   sentiment_cols,
    }


def apply_news_nan_strategy(
    X_asset_sentiment: np.ndarray,   # [T, n_sent_feats]  raw sentiment values for one asset
    news_nan_flag:     np.ndarray,   # [T]                1 = data missing that day
    strategy:          str,          # 'zero_fill' | 'carry_forward' | 'mask_asset'
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply the chosen NaN strategy to raw sentiment values.

    Returns:
      X_out   : [T, n_sent_feats]  cleaned sentiment values
      nan_mask: [T]                bool, True where asset should be masked from loss
                                   (only non-empty when strategy == 'mask_asset')
    """
    missing      = news_nan_flag.astype(bool)  # True = news outlet not available
    X_out        = X_asset_sentiment.copy().astype(np.float32)
    loss_nan_mask = np.zeros(len(news_nan_flag), dtype=bool)

    if strategy == "zero_fill":
        # Replace missing days with 0 (neutral sentiment) — model sees 0
        X_out[missing] = 0.0

    elif strategy == "carry_forward":
        # Forward-fill from last available day, then zero-fill any remaining leading NaNs
        X_out[missing] = np.nan
        df_tmp = pd.DataFrame(X_out)
        df_tmp = df_tmp.ffill().fillna(0.0)
        X_out  = df_tmp.to_numpy(dtype=np.float32)

    elif strategy == "mask_asset":
        # Zero-fill the feature but also mask the asset out of the loss on missing days
        X_out[missing] = 0.0
        loss_nan_mask   = missing

    else:
        raise ValueError(f"Unknown news_nan_strategy: {strategy!r}")

    return X_out, loss_nan_mask

## 8) Stationary transforms  *(extended for sentiment)*

In [ ]:
def _winsorize_fit(x: np.ndarray, q: float) -> Tuple[float, float]:
    lo = np.nanquantile(x, q)
    hi = np.nanquantile(x, 1.0 - q)
    if not np.isfinite(lo): lo = 0.0
    if not np.isfinite(hi): hi = 0.0
    if hi < lo: hi = lo
    return float(lo), float(hi)


def _winsorize_apply(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


@dataclass
class FeatureTransformSpec:
    kind: str
    winsor_q: float = 0.01
    clip_value: float = 8.0


@dataclass
class FittedTransform:
    spec: FeatureTransformSpec
    lo: float = 0.0
    hi: float = 0.0
    center: float = 0.0
    scale: float = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = np.nan_to_num(x, nan=np.nan, posinf=np.nan, neginf=np.nan)

        if self.spec.kind in ("robust_z", "log1p_robust_z"):
            x = _winsorize_apply(x, self.lo, self.hi)

        if self.spec.kind == "identity":
            y = x
        elif self.spec.kind == "bounded_0_100":
            y = (x - 50.0) / 50.0
        elif self.spec.kind == "log1p_robust_z":
            y = np.sign(x) * np.log1p(np.abs(x))
            y = (y - self.center) / self.scale
        elif self.spec.kind == "robust_z":
            y = (x - self.center) / self.scale
        else:
            raise ValueError(f"Unknown transform kind: {self.spec.kind}")

        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.clip(y, -self.spec.clip_value, self.spec.clip_value)
        return y.astype(np.float32)


def default_transform_spec(feature_name: str) -> FeatureTransformSpec:
    """Assign a normalisation strategy by feature name."""
    name = feature_name.lower()

    # ── Base feature rules (unchanged) ───────────────────────────────────
    if feature_name == "log_ret_1d":
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("rsi", "adx"):
        return FeatureTransformSpec(kind="bounded_0_100", clip_value=2.0)
    if name in ("volume", "volumevariation", "obv", "adi"):
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("open", "high", "low"):
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)

    # ── Sentiment: _NEWS_NAN flag — always identity (binary 0/1) ──────────
    if name == "news_nan":
        return FeatureTransformSpec(kind="identity", clip_value=2.0)

    # ── Sentiment: count features — skewed, large values → log1p ─────────
    if "count_positive" in name or "count_negative" in name or name == "balance":
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)

    # ── Sentiment: ratio/balance_ratio variants — bounded-ish → robust_z ─
    # Covers: balance_ratio, ratio_positive, ratio_negative,
    #         balance_ratio_roll*, balance_ratio_t-*, balance_ratio_variation*
    if ("ratio" in name or "variation" in name or name.startswith("balance_ratio")):
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)

    # ── Default fallback ──────────────────────────────────────────────────
    return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)


def fit_transforms_on_train(
    X_train_raw: np.ndarray,
    feat_names: List[str],
    market_train_raw: Optional[np.ndarray],
) -> Tuple[List[FittedTransform], Optional[FittedTransform]]:
    """Fit per-feature transforms on the training window only. Unchanged logic."""
    fitted = []
    for j, fname in enumerate(feat_names):
        spec = default_transform_spec(fname)
        vals = X_train_raw[:, :, j].reshape(-1)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            fitted.append(FittedTransform(spec=spec))
            continue
        if spec.kind in ("robust_z", "log1p_robust_z"):
            lo, hi  = _winsorize_fit(vals, spec.winsor_q)
            vals_w  = _winsorize_apply(vals, lo, hi)
            if spec.kind == "log1p_robust_z":
                vals_w = np.sign(vals_w) * np.log1p(np.abs(vals_w))
            center  = float(np.nanmedian(vals_w))
            mad     = float(np.nanmedian(np.abs(vals_w - center)))
            scale   = 1.4826 * mad
            if not np.isfinite(scale) or scale < 1e-6:
                scale = float(np.nanstd(vals_w))
            if not np.isfinite(scale) or scale < 1e-6:
                scale = 1.0
            fitted.append(FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale))
        elif spec.kind in ("bounded_0_100", "identity"):
            fitted.append(FittedTransform(spec=spec))
        else:
            raise ValueError(spec.kind)

    market_ft = None
    if market_train_raw is not None:
        spec   = FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
        v      = market_train_raw[:, 0]
        v      = v[np.isfinite(v)]
        lr     = np.log(v[1:] / v[:-1]) if v.size > 2 else np.array([0.0])
        lr     = lr[np.isfinite(lr)]
        lo, hi = _winsorize_fit(lr, spec.winsor_q)
        lr_w   = _winsorize_apply(lr, lo, hi)
        center = float(np.nanmedian(lr_w))
        mad    = float(np.nanmedian(np.abs(lr_w - center)))
        scale  = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-6:
            scale = float(np.nanstd(lr_w))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
        market_ft = FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale)

    return fitted, market_ft


def apply_transforms(
    X_raw: np.ndarray,
    fitted: List[FittedTransform],
    market_raw: Optional[np.ndarray],
    market_ft: Optional[FittedTransform],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = np.empty_like(X_raw, dtype=np.float32)
    for j, ft in enumerate(fitted):
        X[:, :, j] = ft.transform(X_raw[:, :, j])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    m_out = None
    if market_raw is not None and market_ft is not None:
        v      = market_raw[:, 0].astype(np.float32)
        lr     = np.full_like(v, np.nan)
        valid  = np.isfinite(v[1:]) & np.isfinite(v[:-1]) & (v[1:] > 0) & (v[:-1] > 0)
        lr[1:][valid] = np.log(v[1:][valid] / v[:-1][valid])
        lr     = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
        m_out  = market_ft.transform(lr).reshape(-1, 1).astype(np.float32)
        m_out  = np.nan_to_num(m_out, nan=0.0, posinf=0.0, neginf=0.0)

    return X, m_out

## 9) Raw arrays & targets  *(extended for sentiment)*

In [ ]:
def apply_spread_to_target(raw_ret: np.ndarray, spread: float) -> np.ndarray:
    return np.sign(raw_ret) * np.maximum(np.abs(raw_ret) - spread, 0.0)


def build_raw_arrays_with_sentiment(
    df: pd.DataFrame,
    assets: List[str],
    picks: Dict[str, Any],
    features: Dict[str, List[str]],
    logger: logging.Logger,
    sentiment_cols: List[str],        # suffixes e.g. ['news_threshold0.50_balance_ratio']
    news_nan_strategy: str,           # 'zero_fill' | 'carry_forward' | 'mask_asset'
    spreads: Optional[np.ndarray] = None,
):
    """
    Extended version of build_raw_arrays that appends sentiment features and
    the _NEWS_NAN flag to X_raw.

    _NEWS_NAN is a SINGLE GLOBAL column in df (not per-asset). It is read once
    and broadcast identically to every asset's feature slice.

    The NaN strategy is applied globally: when 'mask_asset', ALL assets are
    masked from the loss on news-down days (since the flag is the same for all).

    Returns:
      closes_raw     : [T, N]
      X_raw          : [T, N, F+S+1]  base feats + sentiment feats + _NEWS_NAN flag
      y_model        : [T, N]  spread-adjusted target
      y_raw          : [T, N]  raw close-to-close return
      y_mask         : [T, N]  valid target mask (may be narrowed by mask_asset strategy)
      feat_names     : List[str]  one name per feature dimension in X_raw
      market_raw     : [T, 1] or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)

    # sentiment_cols are per-asset suffixes; _NEWS_NAN is a single global column
    cols = required_df_columns(assets, per_asset_feats, picks,
                                sentiment_feats=sentiment_cols)  # _NEWS_NAN added inside
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f"Missing columns ({len(missing)}): {missing[:25]}"
                     f"{'...' if len(missing) > 25 else ''}")
        raise KeyError("Dataset missing required columns for this feature set.")

    if spreads is None:
        spreads = np.zeros(len(assets), dtype=np.float32)
    else:
        spreads = np.asarray(spreads, dtype=np.float32)
        if spreads.shape[0] != len(assets):
            raise ValueError(f"spreads length {len(spreads)} != {len(assets)}")

    T = len(df)
    N = len(assets)
    F = len(per_asset_feats)   # base features
    S = len(sentiment_cols)    # chosen sentiment indicators
    # +1 for _NEWS_NAN flag — always the last feature dimension
    F_total = F + S + 1

    closes = np.zeros((T, N), dtype=np.float32)
    X_raw  = np.zeros((T, N, F_total), dtype=np.float32)

    # ── Load the global _NEWS_NAN flag ONCE ───────────────────────────────
    news_nan_flag = df["_NEWS_NAN"].to_numpy(np.float32)  # [T], 1 = feed down

    # Apply NaN strategy to sentiment values to get the global loss mask
    # We pass a dummy zero-array for the sentiment values at this stage;
    # we only want the loss mask derived from the flag.
    _, global_nan_loss_mask = apply_news_nan_strategy(
        np.zeros((T, max(S, 1)), dtype=np.float32),
        news_nan_flag,
        news_nan_strategy,
    )  # global_nan_loss_mask: [T] bool

    for i, a in enumerate(assets):
        c = df[f"{a}_Close"].to_numpy(np.float32)
        closes[:, i] = c

        # ── Base features (same logic as v5.2.2) ─────────────────────────
        for j, f in enumerate(per_asset_feats):
            if f == "log_ret_1d":
                lr    = np.full_like(c, np.nan)
                valid = (c[1:] > 0) & (c[:-1] > 0) & np.isfinite(c[1:]) & np.isfinite(c[:-1])
                lr[1:][valid] = np.log(c[1:][valid] / c[:-1][valid])
                X_raw[:, i, j] = lr
            elif f in ("Open", "High", "Low"):
                raw   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(raw, np.nan)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith("SMA"):
                sma   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(sma, np.nan)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f"{a}_{f}"].to_numpy(np.float32)

        # ── Sentiment features (per-asset values, global missingness) ─────
        if S > 0:
            sent_raw = np.stack(
                [df[f"{a}_{sf}"].to_numpy(np.float32) for sf in sentiment_cols],
                axis=1,
            )  # [T, S]
            # Clean the values using the global flag (strategy determines fill)
            sent_clean, _ = apply_news_nan_strategy(
                sent_raw, news_nan_flag, news_nan_strategy
            )
            X_raw[:, i, F:F + S] = sent_clean

        # ── _NEWS_NAN flag — broadcast global flag to every asset ──────────
        X_raw[:, i, F + S] = news_nan_flag

    # ── Targets ──────────────────────────────────────────────────────────
    y_model  = np.full((T, N), np.nan, dtype=np.float32)
    y_raw    = np.full((T, N), np.nan, dtype=np.float32)
    valid_y  = np.zeros((T, N), dtype=bool)

    for i in range(N):
        c      = closes[:, i]
        valid  = (np.isfinite(c[:-1]) & np.isfinite(c[1:]) &
                  (c[:-1] > 0.0) & (c[1:] > 0.0))
        tmp_raw   = np.full(T - 1, np.nan, dtype=np.float32)
        tmp_model = np.full(T - 1, np.nan, dtype=np.float32)
        raw_ret   = (c[1:][valid] / c[:-1][valid]) - 1.0
        model_ret = apply_spread_to_target(raw_ret, float(spreads[i]))
        tmp_raw[valid]   = raw_ret.astype(np.float32)
        tmp_model[valid] = model_ret.astype(np.float32)
        y_raw[:-1, i]    = tmp_raw
        y_model[:-1, i]  = tmp_model
        valid_y[:-1, i]  = valid

    y_raw   = np.where(np.isfinite(y_raw),   y_raw,   np.nan).astype(np.float32)
    y_model = np.where(np.isfinite(y_model), y_model, np.nan).astype(np.float32)

    # Narrow y_mask by global news NaN mask (only non-trivial for mask_asset strategy).
    # global_nan_loss_mask is [T]; broadcast to [T, N] to zero out ALL assets on down days.
    y_mask = valid_y & ~global_nan_loss_mask[:, None]

    market = None
    if picks.get("Market") is not None:
        market = df[picks["Market"]].to_numpy(np.float32).reshape(-1, 1)

    # Full feature name list (for transform dispatch)
    feat_names = list(per_asset_feats) + list(sentiment_cols) + ["_NEWS_NAN"]

    return closes, X_raw, y_model, y_raw, y_mask, feat_names, market


def compute_sample_validity(
    X_raw: np.ndarray,
    y_mask: np.ndarray,
    lookback: int,
    min_valid_targets: int = 1,
) -> np.ndarray:
    T, N, F  = X_raw.shape
    finite_feats = np.isfinite(X_raw).all(axis=2)  # [T, N]
    sample_ok    = np.zeros(T, dtype=bool)
    for t in range(lookback - 1, T - 1):
        win_ok    = finite_feats[t - lookback + 1: t + 1].all()
        target_ok = int(y_mask[t].sum()) >= min_valid_targets
        sample_ok[t] = bool(win_ok and target_ok)
    return sample_ok

## 10) Dataset & DataLoader  *(unchanged from v5.2.2)*

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, y_mask, valid_t, lookback, market=None):
        self.X       = X
        self.y       = y
        self.y_mask  = y_mask
        self.valid_t = np.asarray(valid_t, dtype=np.int64)
        self.lookback= int(lookback)
        self.market  = market

    def __len__(self):
        return len(self.valid_t)

    def __getitem__(self, idx):
        t = int(self.valid_t[idx])
        x = self.X[t - self.lookback + 1: t + 1]  # [L, N, F]
        if self.market is not None:
            m = self.market[t - self.lookback + 1: t + 1]        # [L, 1]
            m = np.repeat(m[:, None, :], x.shape[1], axis=1)     # [L, N, 1]
            x = np.concatenate([x, m], axis=2)
        y = self.y[t].copy()
        m = self.y_mask[t].copy()
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )


def make_loader(X, y, y_mask, valid_t, lookback, batch_size, market, shuffle):
    ds = SequenceDataset(X=X, y=y, y_mask=y_mask, valid_t=valid_t,
                         lookback=lookback, market=market)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## 11) Model — JointAssetLSTM  *(unchanged from v5.2.2)*

In [ ]:
class CrossAssetMixer(nn.Module):
    def __init__(self, d_model, num_heads, dropout, num_layers):
        super().__init__()
        if num_layers <= 0:
            self.net = nn.Identity()
        else:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads,
                dim_feedforward=max(4 * d_model, 64),
                dropout=dropout, batch_first=True,
                activation="gelu", norm_first=True,
            )
            self.net = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x):
        return self.net(x)


class JointAssetLSTM(nn.Module):
    def __init__(self, input_dim_per_asset, num_assets, hidden_size, num_layers,
                 dropout, bidirectional=False, encoder_proj=0,
                 asset_mixer_layers=1, asset_mixer_heads=4,
                 fusion_hidden=64, use_residual_fusion=True):
        super().__init__()
        self.num_assets = num_assets
        lstm_dropout    = dropout if num_layers > 1 else 0.0
        self.encoder    = nn.LSTM(
            input_size=input_dim_per_asset, hidden_size=hidden_size,
            num_layers=num_layers, dropout=lstm_dropout,
            batch_first=True, bidirectional=bidirectional,
        )
        enc_dim = hidden_size * (2 if bidirectional else 1)
        if encoder_proj > 0:
            self.asset_proj = nn.Sequential(
                nn.Linear(enc_dim, encoder_proj), nn.GELU(), nn.Dropout(dropout))
            asset_dim = encoder_proj
        else:
            self.asset_proj = nn.Identity()
            asset_dim       = enc_dim
        if asset_mixer_layers > 0:
            valid_heads = [h for h in [1, 2, 4, 8] if asset_dim % h == 0]
            if not valid_heads:          asset_mixer_heads = 1
            elif asset_mixer_heads not in valid_heads: asset_mixer_heads = max(valid_heads)
        self.asset_mixer = CrossAssetMixer(
            d_model=asset_dim,
            num_heads=asset_mixer_heads if asset_mixer_layers > 0 else 1,
            dropout=dropout, num_layers=asset_mixer_layers,
        )
        self.use_residual_fusion = use_residual_fusion
        fusion_in = num_assets * asset_dim
        if fusion_hidden > 0:
            self.head = nn.Sequential(
                nn.Linear(fusion_in, fusion_hidden), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(fusion_hidden, num_assets))
        else:
            self.head = nn.Linear(fusion_in, num_assets)
        if use_residual_fusion:
            self.residual_head = nn.Linear(asset_dim, 1)

    def forward(self, x):
        B, L, N, F = x.shape
        if N != self.num_assets:
            raise ValueError(f"Expected N={self.num_assets}, got N={N}")
        x      = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, L, F)
        out, _ = self.encoder(x)
        h      = out[:, -1, :]
        h      = self.asset_proj(h)
        h      = h.reshape(B, N, -1)
        z      = self.asset_mixer(h)
        yhat   = self.head(z.reshape(B, -1))
        if self.use_residual_fusion:
            yhat = yhat + self.residual_head(z).squeeze(-1)
        return yhat

## 12) Loss & metrics  *(unchanged from v5.2.2)*

In [ ]:
def masked_mse_loss(pred, target, mask):
    mask        = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe   = torch.where(mask > 0, pred,   torch.zeros_like(pred))
    err2        = (pred_safe - target_safe) ** 2 * mask
    loss        = err2.sum() / torch.clamp(mask.sum(), min=1.0)
    if not torch.isfinite(loss):
        raise ValueError("masked_mse_loss became non-finite.")
    return loss


def masked_huber_loss(pred, target, mask, delta=0.01):
    mask        = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe   = torch.where(mask > 0, pred,   torch.zeros_like(pred))
    err         = pred_safe - target_safe
    abs_err     = torch.abs(err)
    delta_t     = torch.tensor(delta, dtype=pred.dtype, device=pred.device)
    quad        = torch.minimum(abs_err, delta_t)
    loss        = (0.5 * quad ** 2 + delta_t * (abs_err - quad)) * mask
    loss        = loss.sum() / torch.clamp(mask.sum(), min=1.0)
    if not torch.isfinite(loss):
        raise ValueError("masked_huber_loss became non-finite.")
    return loss


def regression_metrics(y_true, y_pred, mask):
    valid = mask.astype(bool)
    yt, yp = y_true[valid], y_pred[valid]
    if yt.size == 0:
        return {k: np.nan for k in
                ["rmse", "mae", "r2", "directional_acc", "ic_pearson", "ic_spearman"]}
    rmse   = float(np.sqrt(np.mean((yp - yt) ** 2)))
    mae    = float(np.mean(np.abs(yp - yt)))
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2     = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan
    dir_acc= float(np.mean(np.sign(yp) == np.sign(yt)))
    if yt.size > 1 and np.std(yt) > 1e-12 and np.std(yp) > 1e-12:
        ic_p   = float(np.corrcoef(yt, yp)[0, 1])
        ry, rp = pd.Series(yt).rank().to_numpy(), pd.Series(yp).rank().to_numpy()
        ic_s   = float(np.corrcoef(ry, rp)[0, 1]) if np.std(ry)>1e-12 and np.std(rp)>1e-12 else np.nan
    else:
        ic_p = ic_s = np.nan
    return {"rmse": rmse, "mae": mae, "r2": r2,
            "directional_acc": dir_acc, "ic_pearson": ic_p, "ic_spearman": ic_s}


def per_asset_metrics(y_true, y_pred, mask, asset_names):
    return [{"asset": a, **regression_metrics(y_true[:, [i]], y_pred[:, [i]], mask[:, [i]])}
            for i, a in enumerate(asset_names)]

## 13) Training loop  *(unchanged from v5.2.2)*

In [ ]:
def evaluate_model(model, loader, device, loss_name="mse", huber_delta=0.01):
    model.eval()
    ys, yhs, ms, ts, losses = [], [], [], [], []
    with torch.no_grad():
        for xb, yb, mb, tb in loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            pred        = model(xb)
            loss        = (masked_huber_loss(pred, yb, mb, delta=huber_delta)
                           if loss_name == "huber" else masked_mse_loss(pred, yb, mb))
            losses.append(float(loss.item()))
            ys.append(yb.cpu().numpy())
            yhs.append(pred.cpu().numpy())
            ms.append(mb.cpu().numpy())
            ts.append(tb.cpu().numpy())
    y_true = np.concatenate(ys,  axis=0) if ys  else np.empty((0, 0), dtype=np.float32)
    y_pred = np.concatenate(yhs, axis=0) if yhs else np.empty((0, 0), dtype=np.float32)
    mask   = np.concatenate(ms,  axis=0) if ms  else np.empty((0, 0), dtype=np.float32)
    t_idx  = np.concatenate(ts,  axis=0) if ts  else np.empty((0,),   dtype=np.int64)
    metrics       = regression_metrics(y_true, y_pred, mask)
    metrics["loss"] = float(np.mean(losses)) if losses else np.nan
    return {"metrics": metrics, "y_true": y_true, "y_pred": y_pred,
            "mask": mask, "t_idx": t_idx}


def train_one_model(X, y, y_mask, market, train_t, es_t, lookback,
                    input_dim_per_asset, output_dim, params, seed, logger):
    seed_everything(seed)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader = make_loader(X, y, y_mask, train_t, lookback,
                               params["batch_size"], market, shuffle=True)
    es_loader    = make_loader(X, y, y_mask, es_t, lookback,
                               params["batch_size"], market, shuffle=False)
    model = JointAssetLSTM(
        input_dim_per_asset=input_dim_per_asset,
        num_assets=output_dim,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
        encoder_proj=params["encoder_proj"],
        asset_mixer_layers=params["asset_mixer_layers"],
        asset_mixer_heads=params["asset_mixer_heads"],
        fusion_hidden=params["fusion_hidden"],
        use_residual_fusion=params["use_residual_fusion"],
    ).to(device)
    optimizer    = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"],
        weight_decay=params["weight_decay"])
    if params["scheduler"] == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(params["max_epochs"], 5))
    elif params["scheduler"] == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)
    else:
        scheduler = None

    best_state, best_score, best_epoch, patience_count = None, np.inf, -1, 0
    for epoch in range(1, params["max_epochs"] + 1):
        model.train()
        batch_losses = []
        for xb, yb, mb, _ in train_loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            if not torch.isfinite(xb).all(): raise ValueError("Non-finite xb.")
            if mb.sum() <= 0: continue
            if not torch.isfinite(yb[mb > 0]).all(): raise ValueError("Non-finite targets.")
            optimizer.zero_grad()
            pred = model(xb)
            if not torch.isfinite(pred).all(): raise ValueError("Non-finite predictions.")
            loss = (masked_huber_loss(pred, yb, mb, delta=params["huber_delta"])
                    if params["loss_name"] == "huber" else masked_mse_loss(pred, yb, mb))
            if not torch.isfinite(loss): raise ValueError("Non-finite loss.")
            loss.backward()
            if params["grad_clip"] and params["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params["grad_clip"])
            optimizer.step()
            batch_losses.append(float(loss.item()))
        train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
        es_eval    = evaluate_model(model, es_loader, device,
                                    params["loss_name"], params["huber_delta"])
        es_loss    = float(es_eval["metrics"]["loss"])
        es_rmse    = float(es_eval["metrics"]["rmse"])
        logger.info(f"[TRAIN] seed={seed} epoch={epoch:03d} "
                    f"train_loss={train_loss:.6f} es_loss={es_loss:.6f} es_rmse={es_rmse:.6f}")
        if scheduler is not None:
            scheduler.step(es_loss) if params["scheduler"] == "plateau" else scheduler.step()
        if np.isfinite(es_loss) and es_loss < (best_score - params["min_delta"]):
            best_score, best_epoch, best_state, patience_count = es_loss, epoch, copy.deepcopy(model.state_dict()), 0
        else:
            patience_count += 1
        if epoch >= params["min_epochs"] and patience_count >= params["patience"]:
            logger.info(f"[EARLY STOP] seed={seed} best_epoch={best_epoch} best_es_loss={best_score:.6f}")
            break
    if best_state is None:
        raise ValueError("No valid best_state found.")
    model.load_state_dict(best_state)
    return model, {}

## 14) One split / one seed  *(unchanged from v5.2.2)*

In [ ]:
def save_cumulative_returns_plot(out_dir, prefix, dates, y_true, y_pred, mask, asset_names):
    safe_makedirs(out_dir)
    plt.figure(figsize=(12, 6))
    any_line = False
    for i, asset in enumerate(asset_names):
        valid = mask[:, i].astype(bool)
        if valid.sum() == 0: continue
        strat_ret = np.sign(y_pred[valid, i]) * y_true[valid, i]
        plt.plot(dates[valid], np.cumprod(1.0 + strat_ret), label=asset)
        any_line = True
    if any_line:
        plt.title(f"{prefix} | cumulative returns by asset")
        plt.xlabel("date"); plt.ylabel("wealth")
        plt.legend(loc="best", ncol=2); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix}_cumulative_returns.png"))
    plt.close()


def save_per_asset_metrics_json(out_path, y_true, y_pred, mask, asset_names):
    with open(out_path, "w") as f:
        json.dump(per_asset_metrics(y_true, y_pred, mask, asset_names), f, indent=2)


def train_and_eval_one_split_seed(
    X_transformed, y_model, y_mask, market_transformed,
    asset_names, dates_index, train_idx, val_idx,
    lookback, seed, model_params, trial_dir, logger, split_i, save_artifacts=False,
):
    train_core_idx, es_idx = split_train_for_early_stop(
        train_idx, dates_index, es_years=1, min_train_days=252)
    sample_valid = compute_sample_validity(
        X_transformed, y_mask, lookback,
        min_valid_targets=max(1, len(asset_names) // 3))
    train_t = train_core_idx[sample_valid[train_core_idx]]
    es_t    = es_idx[sample_valid[es_idx]]
    val_t   = val_idx[sample_valid[val_idx]]
    if len(train_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
        raise optuna.exceptions.TrialPruned()
    if not np.isfinite(X_transformed).all():
        raise ValueError("X_transformed contains non-finite values.")
    if market_transformed is not None and not np.isfinite(market_transformed).all():
        raise ValueError("market_transformed contains non-finite values.")
    input_dim_per_asset = X_transformed.shape[2] + (1 if market_transformed is not None else 0)
    output_dim = len(asset_names)
    logger.info(f"[SPLIT {split_i}] seed={seed} "
                f"train={len(train_t)} es={len(es_t)} val={len(val_t)} "
                f"input_dim={input_dim_per_asset}")
    model, _ = train_one_model(
        X=X_transformed, y=y_model, y_mask=y_mask,
        market=market_transformed,
        train_t=train_t, es_t=es_t,
        lookback=lookback, input_dim_per_asset=input_dim_per_asset,
        output_dim=output_dim, params=model_params, seed=seed, logger=logger,
    )
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    val_loader = make_loader(X_transformed, y_model, y_mask, val_t, lookback,
                             model_params["batch_size"], market_transformed, shuffle=False)
    val_eval   = evaluate_model(model, val_loader, device,
                                model_params["loss_name"], model_params["huber_delta"])
    vm = val_eval["metrics"]
    logger.info(f"[VAL] split={split_i} seed={seed} rmse={vm['rmse']:.6f} "
                f"ic_s={vm['ic_spearman']:.4f}")
    if save_artifacts:
        plot_dir = os.path.join(trial_dir, "artifacts", f"split_{split_i}", f"seed_{seed}")
        safe_makedirs(plot_dir)
        save_cumulative_returns_plot(
            plot_dir, f"split{split_i}_seed{seed}_val",
            dates_index[val_eval["t_idx"]], val_eval["y_true"],
            val_eval["y_pred"], val_eval["mask"], asset_names)
        save_per_asset_metrics_json(
            os.path.join(plot_dir, f"split{split_i}_seed{seed}_per_asset_metrics.json"),
            val_eval["y_true"], val_eval["y_pred"], val_eval["mask"], asset_names)
    return {"metrics": vm, "y_true": val_eval["y_true"], "y_pred": val_eval["y_pred"],
            "mask": val_eval["mask"], "t_idx": val_eval["t_idx"]}

## 15) Per-LSTM sentiment Optuna objective & study runner

In [ ]:
# Pipeline
#
# For each rank i in 0..TOP_N_BASE-1:
#   - base_config[i] carries frozen technical feature picks from v5.2.2
#   - run N_TRIALS Optuna trials, each searching:
#       sentiment group (live mandatory, hist optional)
#       threshold, NaN strategy, ALL LSTM+training hyperparameters
#   - save to its own DB: v530/lstm_top{i}/lstm_top{i}.db
#   - best trial of each DB = the tuned LSTM for rank i
#
# Then run BL Optuna (N_TRIALS_BL) using the best trial from every DB as the
# ensemble, optimising tau / risk_aversion / omega_scale / max_weight.
# top_n is FIXED to TOP_N_BASE (already determined by v5.2.2 BL study).

def _build_model_params_from_trial(t: optuna.trial.FrozenTrial) -> Dict[str, Any]:
    p = t.params
    return {
        "hidden_size":         p["hidden_size"],
        "num_layers":          p["num_layers"],
        "dropout":             p["dropout"],
        "bidirectional":       bool(p["bidirectional"]),
        "encoder_proj":        p["encoder_proj"],
        "asset_mixer_layers":  p["asset_mixer_layers"],
        "asset_mixer_heads":   p["asset_mixer_heads"],
        "fusion_hidden":       p["fusion_hidden"],
        "use_residual_fusion": bool(p["use_residual_fusion"]),
        "scheduler":           p["scheduler"],
        "learning_rate":       p["learning_rate"],
        "weight_decay":        p["weight_decay"],
        "batch_size":          p["batch_size"],
        "max_epochs":          p["max_epochs"],
        "patience":            p["patience"],
        "min_epochs":          p["min_epochs"],
        "grad_clip":           p["grad_clip"],
        "loss_name":           p["loss_name"],
        "huber_delta":         p["huber_delta"],
        "min_delta":           p["min_delta"],
    }


def per_lstm_objective_factory(
    df: pd.DataFrame,
    assets: List[str],
    base_config: Dict[str, Any],   # single frozen config for this rank
    base_log_dir: str,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
):
    """
    Objective for ONE specific top-N LSTM rank.
    The technical feature picks (picks / per_asset_feats) are frozen from base_config.
    Each trial searches: sentiment group + threshold + NaN strategy + all LSTM/training HPs.
    """
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)
    picks       = base_config["picks"]
    per_asset_feats = base_config["per_asset_feats"]

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f"trial_{trial.number:05d}")
        logger    = setup_logger(trial_dir)
        logger.info("=" * 100)
        logger.info(f"TRIAL {trial.number} | base v5.2.2 trial #{base_config['trial_number']} "
                    f"IC={base_config['value']:.5f}")
        logger.info(f"Frozen picks: {picks}  feats: {per_asset_feats}")

        # ── Sentiment search space ────────────────────────────────────────
        sent_picks     = pick_sentiment_features(trial)
        sentiment_cols = sent_picks["sentiment_cols"]
        nan_strategy   = trial.suggest_categorical(
            "news_nan_strategy", ["zero_fill",])

        trial.set_user_attr("sentiment_picks",   sent_picks)
        trial.set_user_attr("news_nan_strategy", nan_strategy)
        trial.set_user_attr("frozen_picks",      picks)
        trial.set_user_attr("frozen_feats",      per_asset_feats)
        trial.set_user_attr("base_trial_number", base_config["trial_number"])
        logger.info(f"Sentiment: {sent_picks}  NaN strategy: {nan_strategy}")

        # ── LSTM + training hyperparameters ───────────────────────────────
        lookback    = trial.suggest_categorical("lookback",    [5, 10, 22])
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        num_layers  = trial.suggest_int("num_layers", 1, 2)
        dropout     = trial.suggest_float("dropout", 0.0, 0.3)
        bidirectional      = bool(trial.suggest_categorical("bidirectional",      [0, 1]))
        encoder_proj       = trial.suggest_categorical("encoder_proj",       [0, 32, 64])
        asset_mixer_layers = trial.suggest_categorical("asset_mixer_layers", [0, 1, 2])
        asset_mixer_heads  = trial.suggest_categorical("asset_mixer_heads",  [1, 2, 4])
        fusion_hidden      = trial.suggest_categorical("fusion_hidden",      [0, 64, 128])
        use_residual_fusion= bool(trial.suggest_categorical("use_residual_fusion", [0, 1]))
        scheduler    = trial.suggest_categorical("scheduler",    ["none", "plateau"])
        learning_rate= trial.suggest_float("learning_rate", 1e-4, 2e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay",  1e-6, 1e-3, log=True)
        batch_size   = trial.suggest_categorical("batch_size",   [32, 64, 128])
        max_epochs   = trial.suggest_categorical("max_epochs",   [20, 30, 40])
        patience     = trial.suggest_categorical("patience",     [5, 7])
        min_epochs   = trial.suggest_categorical("min_epochs",   [5, 8])
        grad_clip    = trial.suggest_categorical("grad_clip",    [0.5, 1.0, 2.0])
        loss_name    = trial.suggest_categorical("loss_name",    ["mse", "huber"])
        huber_delta  = trial.suggest_categorical("huber_delta",  [0.005, 0.01, 0.02])
        min_delta    = trial.suggest_categorical("min_delta",    [0.0, 1e-5])

        model_params = {
            "hidden_size": hidden_size, "num_layers": num_layers, "dropout": dropout,
            "bidirectional": bidirectional, "encoder_proj": encoder_proj,
            "asset_mixer_layers": asset_mixer_layers, "asset_mixer_heads": asset_mixer_heads,
            "fusion_hidden": fusion_hidden, "use_residual_fusion": use_residual_fusion,
            "scheduler": scheduler, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "batch_size": batch_size,
            "max_epochs": max_epochs, "patience": patience, "min_epochs": min_epochs,
            "grad_clip": grad_clip, "loss_name": loss_name,
            "huber_delta": huber_delta, "min_delta": min_delta,
        }

        # ── Build arrays and run walk-forward folds ───────────────────────
        try:
            _, X_raw, y_model, _, y_mask, feat_names, market_raw = \
                build_raw_arrays_with_sentiment(
                    df=df, assets=assets, picks=picks, features=FEATURES,
                    logger=logger, sentiment_cols=sentiment_cols,
                    news_nan_strategy=nan_strategy, spreads=spreads,
                )
        except KeyError as e:
            logger.info(f"[PRUNE] Missing columns: {e}")
            raise optuna.TrialPruned()

        all_rmse, all_ic, all_scores = [], [], []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info("-" * 100)
            logger.info(f"[SPLIT {split_i}] "
                        f"train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | "
                        f"val   {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}")

            X_train_raw      = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            split_seed_scores = []
            for seed in seeds:
                try:
                    res = train_and_eval_one_split_seed(
                        X_transformed=X_trans, y_model=y_model, y_mask=y_mask,
                        market_transformed=market_trans, asset_names=assets,
                        dates_index=dates_index, train_idx=train_idx, val_idx=val_idx,
                        lookback=lookback, seed=seed, model_params=model_params,
                        trial_dir=trial_dir, logger=logger,
                        split_i=split_i, save_artifacts=False,
                    )
                except ValueError as e:
                    logger.info(f"[PRUNE] Non-finite: {e}")
                    raise optuna.TrialPruned()

                ic_s = float(res["metrics"]["ic_spearman"]) \
                    if np.isfinite(res["metrics"]["ic_spearman"]) else -999.0
                rmse = float(res["metrics"]["rmse"])
                all_rmse.append(rmse); all_ic.append(ic_s)
                all_scores.append(ic_s); split_seed_scores.append(ic_s)
                logger.info(f"[SPLIT {split_i}][SEED {seed}] ic_s={ic_s:.6f} rmse={rmse:.6f}")

            split_mean = float(np.mean(split_seed_scores))
            trial.report(split_mean, step=split_i)
            if trial.should_prune():
                logger.info("[PRUNE] Pruned by MedianPruner.")
                raise optuna.TrialPruned()

        final_score = float(np.mean(all_scores))
        trial.set_user_attr("mean_ic",   float(np.mean(all_ic)))
        trial.set_user_attr("mean_rmse", float(np.mean(all_rmse)))
        trial.set_user_attr("all_scores", [float(x) for x in all_scores])
        logger.info(f"TRIAL {trial.number} DONE | score={final_score:.6f} "
                    f"mean_rmse={np.mean(all_rmse):.6f}")
        return final_score

    return objective


def run_per_lstm_search(
    df: pd.DataFrame,
    assets: List[str],
    base_configs: List[Dict[str, Any]],  # top-N configs, index = rank
    out_dir: str,
    n_trials: int,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
) -> List[optuna.Study]:
    """
    Run one independent Optuna study per rank.
    DB saved to: {out_dir}/lstm_top{rank}/lstm_top{rank}.db
    Returns list of studies in rank order.
    """
    studies = []
    for rank, base_config in enumerate(base_configs):
        rank_dir     = os.path.join(out_dir, f"lstm_top{rank}")
        safe_makedirs(rank_dir)
        db_path      = os.path.join(rank_dir, f"lstm_top{rank}.db")
        storage      = f"sqlite:///{db_path}"
        study_name   = f"lstm_top{rank}"

        print(f"\n{'='*70}")
        print(f"Rank {rank} | v5.2.2 trial #{base_config['trial_number']} "
              f"IC={base_config['value']:.5f}")
        print(f"  feats: {base_config['per_asset_feats']}")
        print(f"  DB: {db_path}")

        study = optuna.create_study(
            study_name=study_name,
            direction="maximize",
            storage=storage,
            load_if_exists=True,
            sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        )

        objective = per_lstm_objective_factory(
            df=df, assets=assets, base_config=base_config,
            base_log_dir=rank_dir, seeds=seeds, spreads=spreads,
        )
        study.optimize(objective, n_trials=n_trials, gc_after_trial=True,
                       show_progress_bar=True)

        best = study.best_trial
        with open(os.path.join(rank_dir, "best_trial.json"), "w") as f:
            json.dump({
                "rank":             rank,
                "best_value":       best.value,
                "best_params":      best.params,
                "best_user_attrs":  {k: v for k, v in best.user_attrs.items()
                                     if isinstance(v, (str, int, float, list, dict, bool))},
                "n_trials":         len(study.trials),
            }, f, indent=2)

        print(f"  Best IC: {best.value:.5f} | "
              f"sentiment={best.user_attrs.get('sentiment_picks',{}).get('sentiment_cols')} | "
              f"nan={best.user_attrs.get('news_nan_strategy')}")
        studies.append(study)

    return studies


# ─── BL helpers (reused from v5.2.2 pattern) ────────────────────────────────

def _build_topn_ensemble_for_bl(
    df: pd.DataFrame,
    assets: List[str],
    lstm_studies: List[optuna.Study],  # one per rank, best trial used
    split_train_idx: np.ndarray,
    split_val_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    seed: int,
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    For each rank study, take its best trial, refit on split_train_idx,
    predict on split_val_idx, ensemble (nanmean) across ranks.
    Returns ensemble predictions + covariance needed by BL.
    """
    T = len(df)
    N = len(assets)
    pred_full_list   = []
    canonical_y_raw  = None
    canonical_y_mask = None

    for rank, study in enumerate(lstm_studies):
        best = study.best_trial
        picks          = best.user_attrs["frozen_picks"]
        sentiment_cols = best.user_attrs["sentiment_picks"]["sentiment_cols"]
        nan_strategy   = best.user_attrs["news_nan_strategy"]
        lookback       = int(best.params["lookback"])
        model_params   = _build_model_params_from_trial(best)

        _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = \
            build_raw_arrays_with_sentiment(
                df=df, assets=assets, picks=picks, features=FEATURES,
                logger=logger, sentiment_cols=sentiment_cols,
                news_nan_strategy=nan_strategy, spreads=spreads,
            )

        if canonical_y_raw is None:
            canonical_y_raw  = y_raw.copy()
            canonical_y_mask = y_mask.copy()

        X_train_raw      = X_raw[split_train_idx]
        market_train_raw = market_raw[split_train_idx] if market_raw is not None else None
        fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
        X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

        sample_valid = compute_sample_validity(
            X_trans, y_mask, lookback, min_valid_targets=max(1, N // 3))
        train_core_idx, es_idx = split_train_for_early_stop(
            split_train_idx, dates_index, es_years=1, min_train_days=252)
        train_core_t = train_core_idx[sample_valid[train_core_idx]]
        es_t         = es_idx[sample_valid[es_idx]]
        val_t        = split_val_idx[sample_valid[split_val_idx]]

        if len(train_core_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
            logger.warning(f"Rank {rank}: too few samples, skipping.")
            continue

        input_dim = X_trans.shape[2] + (1 if market_trans is not None else 0)
        model, _  = train_one_model(
            X=X_trans, y=y_model, y_mask=y_mask, market=market_trans,
            train_t=train_core_t, es_t=es_t, lookback=lookback,
            input_dim_per_asset=input_dim, output_dim=N,
            params=model_params, seed=seed, logger=logger,
        )

        device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        val_eval  = evaluate_model_on_t_indices(
            model=model, X=X_trans, y=y_model, y_mask=y_mask,
            market=market_trans, t_idx=val_t, lookback=lookback,
            batch_size=model_params["batch_size"],
            loss_name=model_params["loss_name"],
            huber_delta=model_params["huber_delta"],
        )

        pred_full = np.full((T, N), np.nan, dtype=np.float32)
        if len(val_eval["t_idx"]) > 0:
            pred_full[val_eval["t_idx"]] = val_eval["y_pred"]
        pred_full_list.append(pred_full)
        logger.info(f"Rank {rank}: val IC_s={val_eval['metrics']['ic_spearman']:.4f}")

    if not pred_full_list:
        raise ValueError("No usable predictions from any rank.")

    ensemble = nanmean_predictions(pred_full_list)
    train_cov = estimate_train_covariance(
        canonical_y_raw[split_train_idx],
        canonical_y_mask[split_train_idx],
    )
    return {"ensemble_pred": ensemble, "y_raw": canonical_y_raw,
            "y_mask": canonical_y_mask, "train_cov": train_cov}


def evaluate_model_on_t_indices(
    model, X, y, y_mask, market, t_idx, lookback, batch_size, loss_name, huber_delta
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loader = make_loader(X, y, y_mask, t_idx, lookback, batch_size, market, shuffle=False)
    return evaluate_model(model, loader, device, loss_name, huber_delta)


def nanmean_predictions(pred_list):
    return np.nanmean(np.stack(pred_list, axis=0), axis=0).astype(np.float32)


def run_bl_search(
    df: pd.DataFrame,
    assets: List[str],
    lstm_studies: List[optuna.Study],
    out_dir: str,
    n_trials_bl: int,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_NEWS,
    spreads: Optional[np.ndarray] = None,
) -> optuna.Study:
    """
    BL Optuna search. top_n is fixed = len(lstm_studies) (already set by v5.2.2 BL).
    Searches: tau, risk_aversion, omega_scale, max_weight.
    Saves to: {out_dir}/black_litterman_optuna.db
    """
    safe_makedirs(out_dir)
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)
    db_path     = os.path.join(out_dir, "black_litterman_optuna.db")
    storage     = f"sqlite:///{db_path}"

    def bl_objective(trial: optuna.Trial) -> float:
        tau           = trial.suggest_float("tau",           1e-3, 0.5, log=True)
        risk_aversion = trial.suggest_float("risk_aversion", 0.5,  10.0, log=True)
        omega_scale   = trial.suggest_float("omega_scale",   0.01, 2.0,  log=True)
        max_weight    = trial.suggest_float("max_weight",    0.10, 1.0)
        long_only     = True  # fixed — only long portfolios

        logger = setup_logger(os.path.join(out_dir, f"bl_trial_{trial.number:05d}"))
        logger.info(f"BL TRIAL {trial.number} | tau={tau:.4f} ra={risk_aversion:.4f} "
                    f"omega={omega_scale:.4f} max_w={max_weight:.4f}")

        split_sharpes = []
        for split_i, (train_idx, val_idx) in enumerate(splits):
            try:
                bundle = _build_topn_ensemble_for_bl(
                    df=df, assets=assets, lstm_studies=lstm_studies,
                    split_train_idx=train_idx, split_val_idx=val_idx,
                    dates_index=dates_index, seed=seed, logger=logger,
                    spreads=spreads,
                )
            except Exception as e:
                logger.info(f"[PRUNE] Ensemble failed on split {split_i}: {e}")
                raise optuna.TrialPruned()

            ensemble = bundle["ensemble_pred"]
            y_raw    = bundle["y_raw"]
            y_mask   = bundle["y_mask"]
            cov      = bundle["train_cov"]

            val_pred_mask = y_mask[val_idx] & np.isfinite(ensemble[val_idx])
            bl_res = run_black_litterman_backtest(
                dates_target=dates_index[val_idx + 1],
                y_true=y_raw[val_idx], y_pred=ensemble[val_idx],
                mask=val_pred_mask, asset_names=assets,
                train_cov=cov, tau=tau, risk_aversion=risk_aversion,
                omega_scale=omega_scale, long_only=long_only,
                max_weight=max_weight,
            )
            sharpe = bl_res["metrics"]["sharpe"]
            if not np.isfinite(sharpe): sharpe = -999.0
            split_sharpes.append(float(sharpe))
            logger.info(f"[BL SPLIT {split_i}] sharpe={sharpe:.4f} "
                        f"ann_ret={bl_res['metrics']['ann_return']:.4f}")
            trial.report(float(np.mean(split_sharpes)), step=split_i)
            if trial.should_prune():
                raise optuna.TrialPruned()

        final = float(np.mean(split_sharpes))
        trial.set_user_attr("mean_val_sharpe", final)
        logger.info(f"BL TRIAL {trial.number} DONE | mean_sharpe={final:.4f}")
        return final

    bl_study = optuna.create_study(
        study_name="black_litterman_hpo",
        direction="maximize",
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    bl_study.optimize(bl_objective, n_trials=n_trials_bl, gc_after_trial=True,
                      show_progress_bar=True)

    best = bl_study.best_trial
    with open(os.path.join(out_dir, "best_bl_trial.json"), "w") as f:
        json.dump({"best_value": best.value, "best_params": best.params,
                   "n_trials": len(bl_study.trials)}, f, indent=2)
    print(f"\n=== BEST BL TRIAL ===")
    print(f"Mean val Sharpe: {best.value:.4f}")
    print(f"Params: {best.params}")
    print(f"DB: {db_path}")
    return bl_study

## 16) Search space inspector  *(unchanged from v5.2.2)*

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    if not study.trials:
        raise ValueError("Run at least 1 trial first so distributions are populated.")
    distributions         = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params     = []
    print("Search space:")
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f"  {name:35s}  categorical  {n:>4d} choices")
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f"  {name:35s}  int          {n:>4d} levels  [{dist.low} .. {dist.high}]")
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = "log" if dist.log else "linear"
            print(f"  {name:35s}  float ({scale:6s})  [{dist.low:.2e} .. {dist.high:.2e}]  ← continuous")
    import math
    suggested = math.ceil(math.sqrt(discrete_combinations))
    print()
    print(f"Discrete combinations : {discrete_combinations}")
    print(f"Continuous params     : {continuous_params}")
    print(f"Suggested N_TRIALS    : ceil(sqrt({discrete_combinations})) = {suggested}")
    return suggested

## 17) Black–Litterman  *(unchanged from v5.2.2 — copied for self-containment)*

In [ ]:
def _safe_float(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def get_top_n_completed_trials(study: optuna.Study, top_n: int = 5):
    trials = [t for t in study.trials
              if t.state == TrialState.COMPLETE
              and t.value is not None and np.isfinite(t.value)]
    if not trials:
        raise ValueError("No completed Optuna trials found.")
    return sorted(trials, key=lambda t: t.value, reverse=True)[:top_n]


def make_final_train_test_split_by_target_date(
    dates_index, train_target_end="2023-06-30", test_target_start="2023-07-01"
):
    anchor_idx   = np.arange(len(dates_index) - 1, dtype=np.int64)
    target_dates = dates_index[1:]
    train_idx    = anchor_idx[target_dates <= pd.Timestamp(train_target_end)]
    test_idx     = anchor_idx[target_dates >= pd.Timestamp(test_target_start)]
    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError(f"Empty final split: train={len(train_idx)} test={len(test_idx)}")
    return train_idx, test_idx


def estimate_train_covariance(y_train, y_mask_train, ridge=1e-5):
    y_df   = pd.DataFrame(np.where(y_mask_train, y_train, np.nan))
    Sigma  = y_df.cov(min_periods=max(20, len(y_df) // 10)).to_numpy(dtype=np.float64)
    Sigma  = np.nan_to_num(Sigma, nan=0.0, posinf=0.0, neginf=0.0)
    diag   = np.diag(Sigma).copy()
    pos    = diag[np.isfinite(diag) & (diag > 0)]
    fill   = float(np.median(pos)) if len(pos) > 0 else 1e-4
    for i in range(len(diag)):
        if not np.isfinite(Sigma[i, i]) or Sigma[i, i] <= 0:
            Sigma[i, i] = fill
    Sigma  = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(Sigma.shape[0])
    return Sigma


def black_litterman_posterior(
    Sigma, q, market_weights=None, tau=0.05, risk_aversion=2.5,
    omega_scale=0.25, ridge=1e-8,
):
    N       = Sigma.shape[0]
    Sigma   = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(N)
    w_mkt   = (np.full(N, 1.0 / N) if market_weights is None
               else np.clip(market_weights, 0, None))
    if w_mkt.sum() <= 0: w_mkt = np.full(N, 1.0 / N)
    else: w_mkt = w_mkt / w_mkt.sum()
    q       = np.asarray(q, dtype=np.float64)
    pi      = risk_aversion * (Sigma @ w_mkt)
    P       = np.eye(N)
    tauSig  = tau * Sigma
    Omega   = np.diag(np.maximum(np.diag(tauSig) * omega_scale, ridge))
    A       = np.linalg.inv(tauSig) + P.T @ np.linalg.inv(Omega) @ P
    b       = np.linalg.inv(tauSig) @ pi + P.T @ np.linalg.inv(Omega) @ q
    return np.linalg.solve(A, b), pi, Omega


def mean_variance_weights(
    mu, Sigma, risk_aversion=2.5, long_only=True, max_weight=None, ridge=1e-8
):
    A = risk_aversion * Sigma + ridge * np.eye(len(mu))
    try: w = np.linalg.solve(A, mu)
    except np.linalg.LinAlgError: w = np.full(len(mu), 1.0 / len(mu))
    if long_only: w = np.clip(w, 0, None)
    if max_weight and max_weight > 0: w = np.minimum(w, max_weight)
    return w / w.sum() if w.sum() > 0 else np.full(len(mu), 1.0 / len(mu))


def portfolio_metrics_from_returns(rets, freq=252):
    rets = np.asarray(rets)[np.isfinite(rets)]
    if rets.size == 0:
        return {k: np.nan for k in ["n_days","mean_daily_return","vol_daily",
                                    "ann_return","ann_vol","sharpe","cum_return","max_drawdown"]}
    ann_r  = float(np.mean(rets) * freq)
    ann_v  = float(np.std(rets) * np.sqrt(freq))
    cum    = float(np.prod(1 + rets) - 1)
    roll   = np.cumprod(1 + rets)
    dd     = float(np.min(roll / np.maximum.accumulate(roll)) - 1)
    return {"n_days": len(rets), "mean_daily_return": float(np.mean(rets)),
            "vol_daily": float(np.std(rets)), "ann_return": ann_r,
            "ann_vol": ann_v, "sharpe": ann_r / ann_v if ann_v > 1e-8 else np.nan,
            "cum_return": cum, "max_drawdown": dd}

def run_black_litterman_backtest(
    dates_target: pd.DatetimeIndex,
    y_true: np.ndarray,                 # raw realized returns [T,N]
    y_pred: np.ndarray,                 # model predictions [T,N]
    mask: np.ndarray,                   # [T,N]
    asset_names: List[str],
    train_cov: np.ndarray,
    tau: float = 0.05,
    risk_aversion: float = 2.5,
    omega_scale: float = 0.25,
    long_only: bool = True,
    max_weight: Optional[float] = None,
) -> Dict[str, Any]:
    T, N = y_true.shape
    weights_full = np.zeros((T, N), dtype=np.float64)
    port_rets = np.full(T, np.nan, dtype=np.float64)

    for t in range(T):
        active = mask[t].astype(bool) & np.isfinite(y_pred[t]) & np.isfinite(y_true[t])
        if active.sum() == 0:
            continue

        idx = np.where(active)[0]
        q = y_pred[t, idx].astype(np.float64)
        Sigma_sub = train_cov[np.ix_(idx, idx)]

        mu_bl, _, _ = black_litterman_posterior(
            Sigma=Sigma_sub,
            q=q,
            market_weights=None,
            tau=tau,
            risk_aversion=risk_aversion,
            omega_scale=omega_scale,
        )

        w_sub = mean_variance_weights(
            mu=mu_bl,
            Sigma=Sigma_sub,
            risk_aversion=risk_aversion,
            long_only=long_only,
            max_weight=max_weight,
        )

        w = np.zeros(N, dtype=np.float64)
        w[idx] = w_sub
        weights_full[t] = w

        realized = y_true[t].astype(np.float64)
        port_rets[t] = float(np.dot(w, np.nan_to_num(realized, nan=0.0)))

    weights_df = pd.DataFrame(weights_full, index=dates_target, columns=asset_names)
    returns_df = pd.DataFrame(
        {
            "portfolio_return": port_rets,
            "portfolio_wealth": np.cumprod(1.0 + np.nan_to_num(port_rets, nan=0.0)),
        },
        index=dates_target,
    )

    if len(weights_df) > 1:
        turnover = np.abs(weights_df.diff().fillna(0.0)).sum(axis=1)
        avg_turnover = float(turnover.mean())
    else:
        avg_turnover = np.nan

    pm = portfolio_metrics_from_returns(port_rets[np.isfinite(port_rets)])
    pm["avg_daily_turnover"] = avg_turnover

    return {
        "metrics": pm,
        "weights_df": weights_df,
        "returns_df": returns_df,
    }

## 18) Final test — refit all LSTMs + BL on test set

In [ ]:
def run_final_test(
    df: pd.DataFrame,
    assets: List[str],
    lstm_studies: List[optuna.Study],
    bl_study: optuna.Study,
    out_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_NEWS,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Final out-of-sample evaluation with mid-test refit, mirroring v5.2.2.

    Block 1: train up to 2023-06-30, test 2023-07-01..2024-06-30
    Block 2: train up to 2024-06-30, test 2024-07-01..2025-06-30

    For each block:
      - refit every rank LSTM (best trial from its study) on block train window
      - ensemble predictions
      - apply best BL params → weights and portfolio returns

    Outputs saved to {out_dir}/final_test/:
      test_bl_weights.csv
      test_bl_returns.csv
      test_metrics.json
      train_metrics.json   (val-period metrics from last CV fold, not re-evaluated)
    """
    safe_makedirs(os.path.join(out_dir, "final_test"))
    logger      = setup_logger(os.path.join(out_dir, "final_test"))
    dates_index = pd.DatetimeIndex(df.index)

    best_bl = bl_study.best_trial
    tau           = best_bl.params["tau"]
    risk_aversion = best_bl.params["risk_aversion"]
    omega_scale   = best_bl.params["omega_scale"]
    max_weight    = best_bl.params["max_weight"]
    long_only     = True

    logger.info(f"BL params: tau={tau:.4f} ra={risk_aversion:.4f} "
                f"omega={omega_scale:.4f} max_w={max_weight:.4f}")

    blocks = [
        ("2023-06-30", "2023-07-01", "2024-06-30"),
        ("2024-06-30", "2024-07-01", "2025-06-30"),
    ]

    all_weights_list  = []
    all_returns_list  = []

    for block_i, (train_end, test_start, test_end) in enumerate(blocks):
        logger.info(f"=== BLOCK {block_i} | train→{train_end} "
                    f"test {test_start}..{test_end} ===")

        anchor_idx   = np.arange(len(dates_index) - 1, dtype=np.int64)
        target_dates = dates_index[1:]
        train_idx = anchor_idx[target_dates <= pd.Timestamp(train_end)]
        test_idx  = anchor_idx[
            (target_dates >= pd.Timestamp(test_start)) &
            (target_dates <= pd.Timestamp(test_end))
        ]

        T = len(df)
        N = len(assets)
        pred_full_list   = []
        canonical_y_raw  = None
        canonical_y_mask = None

        for rank, study in enumerate(lstm_studies):
            best = study.best_trial
            picks          = best.user_attrs["frozen_picks"]
            sentiment_cols = best.user_attrs["sentiment_picks"]["sentiment_cols"]
            nan_strategy   = best.user_attrs["news_nan_strategy"]
            lookback       = int(best.params["lookback"])
            model_params   = _build_model_params_from_trial(best)

            _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = \
                build_raw_arrays_with_sentiment(
                    df=df, assets=assets, picks=picks, features=FEATURES,
                    logger=logger, sentiment_cols=sentiment_cols,
                    news_nan_strategy=nan_strategy, spreads=spreads,
                )
            if canonical_y_raw is None:
                canonical_y_raw  = y_raw.copy()
                canonical_y_mask = y_mask.copy()

            X_train_raw      = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            sample_valid = compute_sample_validity(
                X_trans, y_mask, lookback, min_valid_targets=max(1, N // 3))
            train_core_idx, es_idx = split_train_for_early_stop(
                train_idx, dates_index, es_years=1, min_train_days=252)
            train_core_t = train_core_idx[sample_valid[train_core_idx]]
            es_t         = es_idx[sample_valid[es_idx]]
            test_t       = test_idx[sample_valid[test_idx]]

            if len(train_core_t) < 64 or len(es_t) < 32:
                logger.warning(f"Rank {rank} block {block_i}: too few train samples, skipping.")
                continue

            input_dim = X_trans.shape[2] + (1 if market_trans is not None else 0)
            model, _  = train_one_model(
                X=X_trans, y=y_model, y_mask=y_mask, market=market_trans,
                train_t=train_core_t, es_t=es_t, lookback=lookback,
                input_dim_per_asset=input_dim, output_dim=N,
                params=model_params, seed=seed, logger=logger,
            )

            test_eval = evaluate_model_on_t_indices(
                model=model, X=X_trans, y=y_model, y_mask=y_mask,
                market=market_trans, t_idx=test_t, lookback=lookback,
                batch_size=model_params["batch_size"],
                loss_name=model_params["loss_name"],
                huber_delta=model_params["huber_delta"],
            )
            pred_full = np.full((T, N), np.nan, dtype=np.float32)
            if len(test_eval["t_idx"]) > 0:
                pred_full[test_eval["t_idx"]] = test_eval["y_pred"]
            pred_full_list.append(pred_full)
            logger.info(f"Rank {rank} block {block_i}: "
                        f"ic_s={test_eval['metrics']['ic_spearman']:.4f}")

        if not pred_full_list:
            raise ValueError(f"Block {block_i}: no predictions produced.")

        ensemble = nanmean_predictions(pred_full_list)
        train_cov = estimate_train_covariance(
            canonical_y_raw[train_idx], canonical_y_mask[train_idx])

        test_pred_mask = canonical_y_mask[test_idx] & np.isfinite(ensemble[test_idx])
        bl_res = run_black_litterman_backtest(
            dates_target=dates_index[test_idx + 1],
            y_true=canonical_y_raw[test_idx], y_pred=ensemble[test_idx],
            mask=test_pred_mask, asset_names=assets,
            train_cov=train_cov, tau=tau, risk_aversion=risk_aversion,
            omega_scale=omega_scale, long_only=long_only, max_weight=max_weight,
        )
        logger.info(f"Block {block_i} test: {bl_res['metrics']}")
        all_weights_list.append(bl_res["weights_df"])
        all_returns_list.append(bl_res["returns_df"])

    # ── Concatenate both blocks ───────────────────────────────────────────
    weights_df = pd.concat(all_weights_list).sort_index()
    returns_df = pd.concat(all_returns_list).sort_index()

    # Recompute wealth as a running product over the full concatenated period
    port_rets = returns_df["portfolio_return"].values
    returns_df["portfolio_wealth"] = np.cumprod(
        1.0 + np.nan_to_num(port_rets, nan=0.0))

    final_metrics = portfolio_metrics_from_returns(
        port_rets[np.isfinite(port_rets)])
    final_metrics["avg_daily_turnover"] = float(
        np.abs(weights_df.diff().fillna(0.0)).sum(axis=1).mean())

    # ── Save outputs ─────────────────────────────────────────────────────
    out = os.path.join(out_dir, "final_test")
    weights_df.to_csv(os.path.join(out, "test_bl_weights.csv"))
    returns_df.to_csv(os.path.join(out, "test_bl_returns.csv"))
    with open(os.path.join(out, "test_metrics.json"), "w") as f:
        json.dump(final_metrics, f, indent=2)

    print("\n=== FINAL TEST METRICS ===")
    for k, v in final_metrics.items():
        print(f"  {k:30s}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"Weights: {os.path.join(out, 'test_bl_weights.csv')}")
    print(f"Returns: {os.path.join(out, 'test_bl_returns.csv')}")

    return {"metrics": final_metrics, "weights_df": weights_df, "returns_df": returns_df}

## 19) Main execution

In [ ]:
# ─── Load data ───────────────────────────────────────────────────────────────
df = pd.read_parquet(DATASET_PATH)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

with open(SPREADS_PATH) as f:
    spreads_dict = json.load(f)
spreads = np.array([spreads_dict.get(a, 0.0) for a in ASSETS], dtype=np.float32)

print(f"Dataset: {df.shape}  ({df.index[0].date()} → {df.index[-1].date()})")
print(f"Assets:  {ASSETS}")
print(f"Spreads: {dict(zip(ASSETS, spreads.tolist()))}")

In [ ]:
# ─── Sanity-check sentiment columns ──────────────────────────────────────────
_all_live_inds = []
for g in SENTIMENT_LIVE_GROUPS:
    _all_live_inds += list(g) if isinstance(g, tuple) else [g]
_all_hist_inds = []
for g in SENTIMENT_HIST_GROUPS:
    if g == "none": continue
    _all_hist_inds += list(g) if isinstance(g, tuple) else [g]
_spot_inds = _all_live_inds[:2] + _all_hist_inds[:2]

missing_cols = []
# _NEWS_NAN is a single global column — check once
if "_NEWS_NAN" not in df.columns:
    missing_cols.append("_NEWS_NAN")
for a in ASSETS:
    for thr in SENTIMENT_THRESHOLDS:
        for ind in _spot_inds:
            col = f"{a}_news_threshold{thr}_{ind}"
            if col not in df.columns:
                missing_cols.append(col)

if missing_cols:
    print(f"WARNING: {len(missing_cols)} expected sentiment columns not found in df.")
    print("First 10:", missing_cols[:10])
else:
    print("All spot-checked sentiment columns present.")

In [ ]:
# ─── Step 1: tune one LSTM per top-N rank ────────────────────────────────────
# Creates v530/lstm_top{i}/lstm_top{i}.db for each rank i.
# Set N_TRIALS in Config before running.
# Tip: run with N_TRIALS=1 first, then call estimate_search_space(lstm_studies[0]).

safe_makedirs(OUT_DIR)

lstm_studies = run_per_lstm_search(
    df=df,
    assets=ASSETS,
    base_configs=BASE_CONFIGS,
    out_dir=OUT_DIR,
    n_trials=N_TRIALS,
    seeds=MODEL_SEEDS,
    spreads=spreads,
)

print(f"\nCompleted {len(lstm_studies)} LSTM studies.")
for rank, s in enumerate(lstm_studies):
    best = s.best_trial
    print(f"  Rank {rank} | best IC={best.value:.5f} "
          f"sentiment={best.user_attrs.get('sentiment_picks',{}).get('sentiment_cols')} "
          f"nan={best.user_attrs.get('news_nan_strategy')}")

In [ ]:
# ─── (Optional) Inspect search space for one rank ────────────────────────────
estimate_search_space(lstm_studies[0])
print(f"You did {len(lstm_studies[0].trials)} to {len(lstm_studies[-1].trials)} trials per LSTM model.")


In [ ]:
# ─── Step 2: tune BL hyperparameters over the ensemble of best LSTMs ─────────
# top_n is fixed = TOP_N_BASE (from v5.2.2 BL study — not re-searched here).
# Searches: tau, risk_aversion, omega_scale, max_weight.

bl_study = run_bl_search(
    df=df,
    assets=ASSETS,
    lstm_studies=lstm_studies,
    out_dir=OUT_DIR,
    n_trials_bl=N_TRIALS_BL,
    seed=BL_SEED,
    spreads=spreads,
)

In [ ]:
estimate_search_space(bl_study)
print(f"You did {len(bl_study.trials)} trials for the BL model.")


In [ ]:
# ─── Step 3: final test with mid-test refit ───────────────────────────────────
# Refits all LSTMs on full train window, runs BL with best params,
# evaluates on out-of-sample test blocks 2023-H2 and 2024-H2.

results = run_final_test(
    df=df,
    assets=ASSETS,
    lstm_studies=lstm_studies,
    bl_study=bl_study,
    out_dir=OUT_DIR,
    seed=BL_SEED,
    spreads=spreads,
)

In [ ]:
weights = results["weights_df"].values

In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()
print(pm.summary(weights))
pm.plot_weights(weights)
pm.plot_cumulative_returns(weights)


In [ ]:
np.save("LSTMvia-returns_news_weights.npy", weights)